In [1]:
# ============================================================================
# KAGGLE GPU EVALUATION SCRIPT FOR UNSEEN DATASETS (TAILORED TO YOUR TREE)
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# 1. SETUP GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("======================================================================")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))
print("======================================================================")

# 2. AUTOMATIC PATH LOCATOR FOR YOUR KAGGLE INPUT TREE
def find_kaggle_path(keywords, is_file=False):
    """Finds path matching all keywords in /kaggle/input or local fallback."""
    for root, dirs, files in os.walk("/kaggle/input"):
        targets = files if is_file else dirs
        for t in targets:
            full_path = os.path.join(root, t)
            if all(k.lower() in full_path.lower() for k in keywords):
                return full_path
    return None

CHECKPOINT_PATH = find_kaggle_path(["crackdualnet_best.pth"], is_file=True) or "crackdualnet_best.pth"
print("Model Checkpoint Path:", CHECKPOINT_PATH)

# Auto-resolve dataset paths based on your screenshot structure
DATASETS_TO_TEST = []

# 1. CFD
cfd_p = find_kaggle_path(["crackforest", "image"])
if cfd_p: DATASETS_TO_TEST.append({"name": "CFD", "path": cfd_p})

# 2. Asphalt_360x360
asphalt_p = find_kaggle_path(["cracked360"])
if asphalt_p: DATASETS_TO_TEST.append({"name": "Asphalt_360x360", "path": os.path.dirname(asphalt_p)})

# 3. DeepCrack
deepcrack_p = find_kaggle_path(["deepcrack", "test_img"])
if deepcrack_p: DATASETS_TO_TEST.append({"name": "DeepCrack", "path": deepcrack_p})

# 4. SUT_Crack
sut_p = find_kaggle_path(["sut-crack", "3-classification"]) or find_kaggle_path(["sut-crack", "classification"])
if sut_p: DATASETS_TO_TEST.append({"name": "SUT_Crack", "path": sut_p})

for ds in DATASETS_TO_TEST:
    print(f"Detected Dataset [{ds['name']}]: {ds['path']}")

IMG_SIZE = 256
BATCH_SIZE = 64  # Fast GPU parallel processing
OUTPUT_DIR = "/kaggle/working/unseen_dataset_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")


# 3. MODEL & ENHANCEMENT PIPELINE
def ieap_enhance(bgr_img):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_enh = clahe.apply(l)
    enhanced = cv2.merge((l_enh, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


class DualBranchCrackNet(nn.Module):
    def __init__(self, num_classes=2, fused_channels=512, fused_size=16, img_size=256):
        super().__init__()
        self.fused_size = fused_size

        resnet = models.resnet50(weights=None)
        self.cnn_stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        cnn_out_channels = 1024

        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(2,),
            img_size=img_size,
        )
        swin_out_channels = self.swin.feature_info.channels()[0]

        self.cnn_proj = nn.Conv2d(cnn_out_channels, fused_channels, kernel_size=1)
        self.swin_proj = nn.Conv2d(swin_out_channels, fused_channels, kernel_size=1)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fused_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
        self.heatmap_conv = nn.Conv2d(fused_channels, 1, kernel_size=1)

    def forward(self, x):
        f_cnn = self.cnn_stem(x)
        f_cnn = self.cnn_proj(f_cnn)
        f_cnn = F.interpolate(f_cnn, size=(self.fused_size, self.fused_size),
                               mode="bilinear", align_corners=False)

        f_swin = self.swin(x)[0]
        if f_swin.shape[1] != self.swin_proj.in_channels:
            f_swin = f_swin.permute(0, 3, 1, 2).contiguous()
        f_swin = self.swin_proj(f_swin)
        f_swin = F.interpolate(f_swin, size=(self.fused_size, self.fused_size),
                                mode="bilinear", align_corners=False)

        fused = f_cnn + f_swin
        logits = self.classifier(fused)
        heatmap = torch.sigmoid(self.heatmap_conv(fused))
        heatmap_full = F.interpolate(heatmap, size=(256, 256), mode="bilinear", align_corners=False)
        return logits, heatmap_full, fused


def load_model(checkpoint_path):
    model = DualBranchCrackNet(num_classes=2, img_size=IMG_SIZE).to(DEVICE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model


class InferenceDataset(Dataset):
    def __init__(self, file_paths, file_labels=None, img_size=256):
        self.file_paths = file_paths
        self.file_labels = file_labels if file_labels is not None else [-1] * len(file_paths)
        self.img_size = img_size
        self.tfm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label = self.file_labels[idx]
        bgr = cv2.imread(img_path)
        if bgr is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        bgr = cv2.resize(bgr, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        bgr = ieap_enhance(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor_img = self.tfm(Image.fromarray(rgb))
        return tensor_img, label, img_path


def detect_dataset_structure(root_dir):
    if not os.path.isdir(root_dir):
        return None, None

    subdirs = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    names_lower = [d.lower() for d in subdirs]

    has_uncracked = any("uncrack" in n for n in names_lower)
    has_cracked_a = any(("crack" in n and "uncrack" not in n) for n in names_lower)
    has_with_crack = any("with crack" in n for n in names_lower)
    has_without_crack = any("without crack" in n for n in names_lower)

    if (has_cracked_a and has_uncracked) or (has_with_crack and has_without_crack):
        classes = sorted(subdirs)
        class_to_idx = {c: i for i, c in enumerate(classes)}
        return "labeled", class_to_idx

    return "unlabeled", None


def collect_images_recursive(root_dir):
    paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


def evaluate_labeled(model, root_dir, class_to_idx, dataset_name):
    paths, labels = [], []
    for c, idx in class_to_idx.items():
        folder = os.path.join(root_dir, c)
        for fname in os.listdir(folder):
            if fname.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(folder, fname))
                labels.append(idx)

    print(f"[{dataset_name}] LABELED dataset. Classes: {class_to_idx}. Total images: {len(paths)}")

    ds = InferenceDataset(paths, labels, IMG_SIZE)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    all_preds, all_labels, all_paths, all_conf = [], [], [], []
    with torch.no_grad():
        for imgs, lbls, paths_batch in loader:
            imgs = imgs.to(DEVICE)
            logits, _, _ = model(imgs)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_preds.extend(preds.tolist())
            all_labels.extend(lbls.numpy().tolist() if torch.is_tensor(lbls) else list(lbls))
            all_paths.extend(paths_batch)
            all_conf.extend(probs.max(axis=1).tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n==================== {dataset_name} RESULTS ====================")
    print(f"[{dataset_name}] Accuracy : {acc:.4f}")
    print(f"[{dataset_name}] Precision: {prec:.4f}")
    print(f"[{dataset_name}] Recall   : {rec:.4f}")
    print(f"[{dataset_name}] F1-score : {f1:.4f}")
    print(f"[{dataset_name}] Confusion matrix:\n", cm)
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    print(classification_report(all_labels, all_preds, target_names=[idx_to_class[i] for i in sorted(idx_to_class)]))

    out_csv = os.path.join(OUTPUT_DIR, f"{dataset_name}_labeled_predictions.csv")
    pd.DataFrame({
        "path": all_paths,
        "true_label": [idx_to_class[l] for l in all_labels],
        "predicted_label": [idx_to_class[p] for p in all_preds],
        "confidence": all_conf,
    }).to_csv(out_csv, index=False)
    print(f"[{dataset_name}] Per-image predictions saved to: {out_csv}\n")


def evaluate_unlabeled(model, root_dir, dataset_name):
    paths = collect_images_recursive(root_dir)
    print(f"[{dataset_name}] UNLABELED dataset. Total images: {len(paths)}")
    if len(paths) == 0:
        print(f"[{dataset_name}] No images found under {root_dir}.")
        return

    ds = InferenceDataset(paths, None, IMG_SIZE)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    all_paths, all_preds, all_conf = [], [], []
    with torch.no_grad():
        for imgs, _, paths_batch in loader:
            imgs = imgs.to(DEVICE)
            logits, _, _ = model(imgs)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_paths.extend(paths_batch)
            all_preds.extend(preds.tolist())
            all_conf.extend(probs.max(axis=1).tolist())

    pred_names = ["Cracked" if p == 0 else "Uncracked" for p in all_preds]
    n_cracked = pred_names.count("Cracked")
    n_uncracked = pred_names.count("Uncracked")

    print(f"\n==================== {dataset_name} RESULTS ====================")
    print(f"[{dataset_name}] Predicted Cracked  : {n_cracked} ({100*n_cracked/len(paths):.1f}%)")
    print(f"[{dataset_name}] Predicted Uncracked: {n_uncracked} ({100*n_uncracked/len(paths):.1f}%)")
    print(f"[{dataset_name}] Mean prediction confidence: {np.mean(all_conf):.4f}")

    out_csv = os.path.join(OUTPUT_DIR, f"{dataset_name}_unlabeled_predictions.csv")
    pd.DataFrame({
        "path": all_paths,
        "predicted_label": pred_names,
        "confidence": all_conf,
    }).to_csv(out_csv, index=False)
    print(f"[{dataset_name}] Per-image predictions saved to: {out_csv}\n")


# 4. RUN ALL EVALUATIONS
model = load_model(CHECKPOINT_PATH)
for ds_cfg in DATASETS_TO_TEST:
    name, path = ds_cfg["name"], ds_cfg["path"]
    print("\n" + "=" * 70)
    print(f"Testing on dataset: {name}  ({path})")
    print("=" * 70)

    structure, class_to_idx = detect_dataset_structure(path)
    if structure is None:
        print(f"[{name}] Directory not found: {path}. Skipping.\n")
        continue
    elif structure == "labeled":
        evaluate_labeled(model, path, class_to_idx, name)
    else:
        evaluate_unlabeled(model, path, name)

print("\n======================================================================")
print("ALL EVALUATIONS COMPLETED!")
print("CSV Output files saved in: /kaggle/working/unseen_dataset_results/")
print("======================================================================")


Device: cuda
GPU Model: Tesla T4
Model Checkpoint Path: /kaggle/input/models/dinjbu/ieap/pytorch/default/1/crackdualnet_best.pth
Detected Dataset [CFD]: /kaggle/input/datasets/dinjbu/asphalt-360x360/CrackForest-dataset-master/CrackForest-dataset-master/image
Detected Dataset [Asphalt_360x360]: /kaggle/input/datasets/dinjbu/asphalt-360x360/Cracked and Uncracked Image Database_360x360/Cracked and Uncracked Image Database_360x360
Detected Dataset [DeepCrack]: /kaggle/input/datasets/dinjbu/asphalt-360x360/DeepCrack-master/DeepCrack-master/dataset/DeepCrack/test_img
Detected Dataset [SUT_Crack]: /kaggle/input/datasets/dinjbu/asphalt-360x360/SUT-Crack/3-Classification

Testing on dataset: CFD  (/kaggle/input/datasets/dinjbu/asphalt-360x360/CrackForest-dataset-master/CrackForest-dataset-master/image)
[CFD] UNLABELED dataset. Total images: 310

==================== CFD RESULTS ====================
[CFD] Predicted Cracked  : 275 (88.7%)
[CFD] Predicted Uncracked: 35 (11.3%)
[CFD] Mean predictio

In [5]:
# ============================================================
    # SDNET2018 — DualBranchCrackNet (LABEL-CORRECTED)
# Cracked = 0, Non-cracked = 1  (matches training convention)
# ============================================================

import os, time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

IMG_SIZE       = 256
BATCH_SIZE     = 64
NUM_WORKERS    = 4
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
OUTPUT_DIR     = "/kaggle/working/unseen_dataset_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
class DualBranchCrackNet(nn.Module):
    def __init__(self, num_classes=2, fused_channels=512, fused_size=16, img_size=256):
        super().__init__()
        self.fused_size = fused_size
        resnet = models.resnet50(weights=None)
        self.cnn_stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3)
        self.swin = timm.create_model("swin_tiny_patch4_window7_224",
            pretrained=False, features_only=True, out_indices=(2,), img_size=img_size)
        swin_ch = self.swin.feature_info.channels()[0]
        self.cnn_proj  = nn.Conv2d(1024, fused_channels, 1)
        self.swin_proj = nn.Conv2d(swin_ch, fused_channels, 1)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(fused_channels, 128), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(128, num_classes))
        self.heatmap_conv = nn.Conv2d(fused_channels, 1, 1)
    def forward(self, x):
        f_cnn = self.cnn_proj(self.cnn_stem(x))
        f_cnn = F.interpolate(f_cnn, (self.fused_size,)*2, mode="bilinear", align_corners=False)
        f_swin = self.swin(x)[0]
        if f_swin.shape[1] != self.swin_proj.in_channels:
            f_swin = f_swin.permute(0,3,1,2).contiguous()
        f_swin = self.swin_proj(f_swin)
        f_swin = F.interpolate(f_swin, (self.fused_size,)*2, mode="bilinear", align_corners=False)
        fused = f_cnn + f_swin
        return self.classifier(fused), torch.sigmoid(self.heatmap_conv(fused)), fused

# ── Load checkpoint ───────────────────────────────────────────────────────────
CKPT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "crackdualnet_best.pth" in files:
        CKPT = os.path.join(root, "crackdualnet_best.pth"); break
assert CKPT, "crackdualnet_best.pth not found"
model = DualBranchCrackNet(num_classes=2, img_size=IMG_SIZE).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE), strict=True)
model.eval()
print(f"✓ Loaded: {CKPT}")

# ── IEAP ─────────────────────────────────────────────────────────────────────
def ieap_enhance(bgr):
    lab=cv2.cvtColor(bgr,cv2.COLOR_BGR2LAB); l,a,b=cv2.split(lab)
    l=cv2.createCLAHE(clipLimit=2.5,tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)),cv2.COLOR_LAB2BGR)

# ── Dataset ───────────────────────────────────────────────────────────────────
class SDNETDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths=paths; self.labels=labels
        self.tfm=transforms.Compose([transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        bgr=cv2.imread(self.paths[i])
        if bgr is None: bgr=np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8)
        bgr=cv2.resize(bgr,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_AREA)
        bgr=ieap_enhance(bgr)
        return self.tfm(Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))), \
               self.labels[i], self.paths[i]

# ── Find folders ──────────────────────────────────────────────────────────────
CRACKED_DIR = "/kaggle/input/datasets/dinjbu/sdnet-pavement/SDNET /Pavements/Cracked"
NONCRK_DIR  = "/kaggle/input/datasets/dinjbu/sdnet-pavement/SDNET /Pavements/Non-cracked"

assert os.path.isdir(CRACKED_DIR), f"Not found: {CRACKED_DIR}"
assert os.path.isdir(NONCRK_DIR),  f"Not found: {NONCRK_DIR}"

paths, labels = [], []

# ── CRITICAL FIX: Cracked=0, Non-cracked=1 (matches training) ────────────────
for f in sorted(os.listdir(CRACKED_DIR)):
    if f.lower().endswith(IMG_EXTENSIONS):
        paths.append(os.path.join(CRACKED_DIR, f))
        labels.append(0)   # ← Cracked = 0

for f in sorted(os.listdir(NONCRK_DIR)):
    if f.lower().endswith(IMG_EXTENSIONS):
        paths.append(os.path.join(NONCRK_DIR, f))
        labels.append(1)   # ← Non-cracked = 1

print(f"Total      : {len(paths)}")
print(f"Cracked(0) : {labels.count(0)}")
print(f"Non-crk(1) : {labels.count(1)}")

# ── Run ───────────────────────────────────────────────────────────────────────
loader = DataLoader(SDNETDataset(paths, labels), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
ap=[]; al=[]; ac=[]; t0=time.time()
with torch.no_grad():
    for imgs, lbs, _ in loader:
        logits,_,_ = model(imgs.to(DEVICE))
        probs=F.softmax(logits,1).cpu().numpy()
        ap.extend(probs.argmax(1).tolist())
        al.extend(lbs.tolist())
        ac.extend(probs.max(1).tolist())

print(f"✓ {len(ap)} processed in {time.time()-t0:.1f}s")

# ── Metrics (pos_label=0 = Cracked) ──────────────────────────────────────────
acc  = accuracy_score(al, ap)
prec,rec,f1,_ = precision_recall_fscore_support(
    al, ap, average='binary', pos_label=0, zero_division=0)
cm = confusion_matrix(al, ap)

print(f"\n======= SDNET2018 Pavements — DualBranchCrackNet (CORRECTED) =======")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}  (for Cracked class)")
print(f"Recall   : {rec:.4f}  (for Cracked class)")
print(f"F1-score : {f1:.4f}")
print(f"Confusion Matrix:\n{cm}")
print(classification_report(al, ap, target_names=["Cracked","Non-cracked"]))

# ── Save ──────────────────────────────────────────────────────────────────────
lm={0:"Cracked",1:"Non-cracked"}
pd.DataFrame({"path":paths,"true_label":[lm[l] for l in al],
              "predicted_label":[lm[p] for p in ap],"confidence":ac})\
  .to_csv(os.path.join(OUTPUT_DIR,"SDNET2018_CORRECTED_predictions.csv"),index=False)
print(f"\nCSV saved.")


Device: cuda
✓ Loaded: /kaggle/input/models/dinjbu/ieap/pytorch/default/1/crackdualnet_best.pth
Total      : 24334
Cracked(0) : 2608
Non-crk(1) : 21726
✓ 24334 processed in 210.1s

======= SDNET2018 Pavements — DualBranchCrackNet (CORRECTED) =======
Accuracy : 0.7746
Precision: 0.2959  (for Cracked class)
Recall   : 0.7998  (for Cracked class)
F1-score : 0.4320
Confusion Matrix:
[[ 2086   522]
 [ 4963 16763]]
              precision    recall  f1-score   support

     Cracked       0.30      0.80      0.43      2608
 Non-cracked       0.97      0.77      0.86     21726

    accuracy                           0.77     24334
   macro avg       0.63      0.79      0.65     24334
weighted avg       0.90      0.77      0.81     24334


CSV saved.
